# Companion Notebook — Subduction Zones in Parameter Space

**ESS 314 · Lecture 28 (Convergent Plate Boundaries)**

This notebook is the interactive companion to Lecture 28. It loads the actual
per-segment data table of **Wirth et al. (2022)** and reproduces the central
result of that review: the classic Ruff–Kanamori (1980) parameters — plate age
and convergence rate — do *not* predict the maximum magnitude of a subduction
zone, whereas the **seismogenic-zone width** (a geometric property) does.

## Learning Objectives

- **[LO-1]** Load the Wirth et al. (2022) subduction database and plot maximum
  magnitude against the classic and geometric parameters.
- **[LO-3]** Quantify the correlations and show that plate age and convergence
  rate are weak predictors while seismogenic width and downdip curvature are
  the strongest.
- **[LO-3]** Compute the thermal parameter $\Phi = A\,v_c\,\sin\delta$ and test
  whether it separates the giant earthquakes from the rest.
- **[LO-3]** Use the seismic-moment relation to estimate the rupture area and
  width required for an $M_w\,9$, and connect it to seismogenic geometry.

## Prerequisites

- Lecture 28 (read first — this notebook assumes its framing).
- The open-data workflow of the Lecture 26 companion material.
- Basic `numpy` / `matplotlib` / `pandas`.


## 0. Setup

The style block is the ESS 314 quality gate: minimum readable font size and the
colourblind-safe Wong (2011) palette.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13,
    "xtick.labelsize": 12, "ytick.labelsize": 12, "legend.fontsize": 11,
    "figure.dpi": 120,
})
BLUE, ORANGE, SKY, GREEN, VERM, PINK, BLACK = (
    "#0072B2", "#E69F00", "#56B4E9", "#009E73", "#D55E00", "#CC79A7", "#000000")
print("Setup complete.")

## 1. The subduction-zone database (Wirth et al. 2022, Table 1)

The table below is **Table 1 of Wirth et al. (2022)**, transcribed verbatim. Each
row is a margin segment that hosted a recorded or palaeoseismic $M \geq 8.5$
earthquake (plus Cascadia 1700). The columns are the parameters the review tests
as possible controls on maximum magnitude:

| column | meaning | units |
|--------|---------|-------|
| `Mw` | maximum observed / palaeoseismic moment magnitude | — |
| `age_Ma` | incoming-plate age at the trench | Ma |
| `conv_mm_yr` | **trench-normal** convergence rate, HS3-NUVEL1A absolute frame | mm/yr |
| `sed_km` | trench sediment thickness | km |
| `dip_deg` | dip of the seismogenic zone | ° |
| `width_km` | downdip width of the seismogenic zone | km |
| `curvature` | downdip curvature ($10^{-8}\,\mathrm{m}^{-1}$; lower = flatter) | — |
| `rough_km` | long-wavelength (80–100 km) incoming-plate roughness | km |
| `coupling` | geodetic coupling coefficient | — |

**Provenance.** Wirth, E. A., Sahakian, V. J., Wallace, L. M. & Melnick, D.
(2022), *Nat. Rev. Earth Environ.* 3, 125–140, doi:10.1038/s43017-021-00245-w
(a US Government work, public domain). Most parameters derive from the SubMap
database (Heuret & Lallemand 2005, http://submap.gm.univ-montp2.fr/); downdip
curvature from Bletery et al. (2016); coupling from the GEM Faulted Earth
project. Sediment "<0.5 km" is entered as 0.4; "–" (unknown) as `NaN`. Note the
convergence rate is the *trench-normal* rate in an *absolute* reference frame —
not the plate-pair relative rate of Lecture 26 — which is why Sumatra–Andaman
reads 3 mm/yr.

In [ ]:
# Wirth et al. (2022), Table 1 — verbatim.
# columns: name, Mw, age_Ma, conv_mm_yr, sed_km, dip_deg, width_km,
#          curvature, rough_km, coupling
zones = pd.DataFrame([
    ("Tohoku 2011",    9.1, 132, 96, 0.4, 18, 161, 1.39, 139, 0.7),
    ("Cascadia 1700",  9.0,   7, 32, 4.0, 11, 127, 0.94,  76, 0.8),
    ("Maule 2010",     8.8,  34, 62, 2.0, 22, 105, 2.04, 138, 0.8),
    ("Valdivia 1960",  9.5,  23, 75, 1.0, 14, 190, 1.86, 215, 0.8),
    ("Atacama 1922",   8.5,  45, 75, 0.4, 22, 105, 1.77, 188, 0.8),
    ("Rat Is. 1965",   8.7,  49, 36, np.nan, 31, 72, 3.63, 536, 0.5),
    ("Alaska 1964",    9.2,  43, 52, 2.0, 15, 180, 0.60, 223, 0.8),
    ("Alaska 1957",    8.6,  55, 61, 2.0, 35,  75, 2.50, 214, 0.5),
    ("Unimak 1946",    8.6,  57, 62, np.nan, 33, 72, 2.41, 145, 0.5),
    ("Nias 2005",      8.6,  43, 28, 4.0, 11, 174, 2.01, 391, 0.8),
    ("Sumatra 2004",   9.1,  73,  3, 3.0,  9, 243, 2.26, 307, 0.7),
    ("Kuril 1963",     8.5, 117, 71, 0.4, 22, 102, 3.31, 224, 0.8),
    ("Kamchatka 1952", 9.0, 105, 77, 0.4, 27, 110, 2.45, 234, 0.8),
    ("Ecuador 1906",   8.8,  12, 55, 3.0, 20, 101, 2.78, 538, 0.8),
], columns=["name", "Mw", "age_Ma", "conv_mm_yr", "sed_km", "dip_deg",
            "width_km", "curvature", "rough_km", "coupling"])

zones["giant"] = zones["Mw"] >= 8.95   # the M~9 club
zones

## 2. Which parameter controls the maximum earthquake?

Plot maximum magnitude against the classic control (plate age) and the geometric
control (seismogenic-zone width), side by side. If the Ruff–Kanamori recipe held,
the giants would line up with age. They do not — but they do line up with width.

In [ ]:
fig, (axA, axW) = plt.subplots(1, 2, figsize=(12.5, 5.2), sharey=True)
for ax, col, xlabel, title in [
    (axA, "age_Ma", "subducting plate age (Ma)", "(a) plate age - no correlation"),
    (axW, "width_km", "seismogenic-zone width (km)", "(b) seismogenic width - strong control"),
]:
    g, ng = zones[zones.giant], zones[~zones.giant]
    ax.scatter(ng[col], ng["Mw"], s=70, c=BLUE, edgecolor=BLACK, lw=0.6,
               alpha=0.85, label="$M_w$ 8.5-8.9", zorder=3)
    ax.scatter(g[col], g["Mw"], s=200, marker="*", c=VERM, edgecolor=BLACK,
               lw=0.8, label=r"$M_w \geq 9.0$", zorder=4)
    for _, r in g.iterrows():
        ax.annotate(r["name"].split()[0], xy=(r[col], r["Mw"]),
                    xytext=(r[col], r["Mw"] + 0.05), ha="center", fontsize=8.5)
    ax.set_xlabel(xlabel); ax.set_title(title); ax.grid(alpha=0.22)
axA.set_ylabel("maximum observed $M_w$")
axW.axvline(75, color="#999999", ls=":", lw=1.2)
axW.axvline(150, color=GREEN, ls="--", lw=1.4)
axW.text(152, 8.55, "$M\\geq9.2$\nonly >150 km", color=GREEN, fontsize=9)
axW.legend(loc="lower right", framealpha=0.95)
fig.suptitle("Plate age does not order $M_w$; seismogenic width does "
             "(Wirth et al. 2022)", y=1.0)
fig.tight_layout(); plt.show()

The giants span the full range of plate age (Cascadia at 7 Ma, Tōhoku at
132 Ma) but every one of them has a seismogenic width above ~110 km. Width is the
discriminator the classic recipe was missing.

### Exercise 1 — Quantify it, and compare to the published values

Compute the correlation of $M_w$ with each parameter. Wirth et al. (2022), Fig. 3,
report (over their *full* $M \geq 8.5$ data set): age $r=0.05$, convergence rate
$r=0.19$, dip $r=0.24$, **downdip curvature $r=0.42$**, **seismogenic width
$r=0.44$**, coupling $r=0.17$, sediment $r=-0.09$, roughness $r=-0.20$.

In [ ]:
wirth_r = {"age_Ma": 0.05, "conv_mm_yr": 0.19, "dip_deg": 0.24,
           "curvature": 0.42, "width_km": 0.44, "coupling": 0.17,
           "sed_km": -0.09, "rough_km": -0.20}
print(f"{'parameter':12s} {'our r (Table 1)':>16s} {'Wirth full-set r':>18s}")
for col, rw in wirth_r.items():
    m = zones[col].notna()
    r = np.corrcoef(zones.loc[m, col], zones.loc[m, "Mw"])[0, 1]
    print(f"{col:12s} {r:>16.2f} {rw:>18.2f}")

# QUESTION (answer as a comment):
# Rank the parameters by |r| in BOTH columns. Which two come out on top? Are they
# the classic recipe (age, rate) or geometric (width, curvature)? Why might our
# 14-segment values differ from Wirth's full-set values? (Small samples give
# unstable r - that instability is itself a reason no single parameter predicts.)

## 3. The thermal parameter

The thermal parameter $\Phi = A\,v_c\,\sin\delta$ combines age, convergence rate,
and dip into a single "how cold does the slab stay" number — the quantitative
version of the strong-coupling expectation. Does it separate the giants? (Dip is
already in the table.)

In [ ]:
zones["Phi"] = (zones["age_Ma"] * zones["conv_mm_yr"]
                * np.sin(np.radians(zones["dip_deg"])))
fig, ax = plt.subplots(figsize=(8.8, 5.4))
g, ng = zones[zones.giant], zones[~zones.giant]
ax.scatter(ng["Phi"], ng["Mw"], s=70, c=BLUE, edgecolor=BLACK, lw=0.6,
           alpha=0.85, label="$M_w$ 8.5-8.9")
ax.scatter(g["Phi"], g["Mw"], s=180, marker="*", c=VERM, edgecolor=BLACK,
           lw=0.7, label=r"$M_w \geq 9.0$")
for _, r in g.iterrows():
    ax.annotate(r["name"].split()[0], xy=(r.Phi, r.Mw),
                xytext=(r.Phi + 150, r.Mw - 0.04), fontsize=8.5)
ax.set_xlabel(r"thermal parameter $\Phi = A\,v_c\,\sin\delta$")
ax.set_ylabel("maximum observed $M_w$")
ax.set_title("Does the thermal parameter separate the giants?")
ax.grid(alpha=0.22); ax.legend()
fig.tight_layout(); plt.show()
print(f"corr(Mw, Phi) = {np.corrcoef(zones['Phi'], zones['Mw'])[0,1]:+.2f}")

Sumatra (very low trench-normal rate) and Cascadia (young, warm) sit at low
$\Phi$ yet are giants; combining the classic parameters into $\Phi$ does not
rescue the recipe. A cold slab is not a prerequisite for a great earthquake.

### Exercise 2 — Width vs. dip

Width and dip are not independent: a shallow dip pushes the downdip
brittle–ductile limit farther from the trench, widening the seismogenic zone.
Make a scatter of `width_km` against `dip_deg`, colour the points by `giant`, and
describe the relationship. Which quadrant (shallow + wide) holds the giants?

In [ ]:
# Exercise 2: your code here.
# Hint: plt.scatter(zones.dip_deg, zones.width_km,
#                   c=zones.giant.map({True: VERM, False: BLUE}))
# Then add the Wirth thresholds: dip < 20 deg and width > 150 km for M>=9.2.


## 4. The geometric controls: width and curvature

Wirth et al. (2022) find the two strongest correlations are with seismogenic
**width** ($r\approx0.44$) and downdip **curvature** ($r\approx0.42$) — flatter,
wider megathrusts host the biggest earthquakes ("mega-earthquakes rupture flat
megathrusts", Bletery et al. 2016). Plot $M_w$ against curvature (lower =
flatter).

In [ ]:
fig, ax = plt.subplots(figsize=(8.8, 5.4))
g, ng = zones[zones.giant], zones[~zones.giant]
ax.scatter(ng["curvature"], ng["Mw"], s=70, c=BLUE, edgecolor=BLACK, lw=0.6,
           alpha=0.85, label="$M_w$ 8.5-8.9")
ax.scatter(g["curvature"], g["Mw"], s=180, marker="*", c=VERM, edgecolor=BLACK,
           lw=0.7, label=r"$M_w \geq 9.0$")
for _, r in g.iterrows():
    ax.annotate(r["name"].split()[0], xy=(r.curvature, r.Mw),
                xytext=(r.curvature + 0.05, r.Mw - 0.04), fontsize=8.5)
ax.set_xlabel(r"downdip curvature ($10^{-8}\,\mathrm{m}^{-1}$;  lower = flatter)")
ax.set_ylabel("maximum observed $M_w$")
ax.set_title("Flatter megathrusts host the largest earthquakes")
ax.grid(alpha=0.22); ax.legend()
fig.tight_layout(); plt.show()
print(f"corr(Mw, curvature) = {np.corrcoef(zones.curvature, zones.Mw)[0,1]:+.2f}"
      f"  (negative: flatter -> bigger)")

Sediment and roughness are **secondary** controls in the review (sediment
$r=-0.09$, roughness $r=-0.20$ over the full set): a smooth, well-sedimented
interface helps, but the $M\,9.1$ Tōhoku margin is sediment-starved yet smooth —
so it is interface *smoothness*, not sediment volume, that matters. Geometry
(width, curvature) leads; sediment, roughness, fluids, and upper-plate structure
follow.

## 5. How big a fault do you need for an $M_w\,9$?

The moment relation $M_0 = \mu\,\bar D\,(L\,W)$ ties magnitude to rupture
**area**. Inverting it shows why seismogenic *width* matters so much: a wide
locked zone supplies the area an $M\,9$ requires.

In [ ]:
def Mw_to_M0(Mw):
    # moment magnitude -> seismic moment (N m)
    return 10 ** (1.5 * (Mw + 6.07))

def rupture_length(Mw, mu=40e9, slip=15.0, width_km=100.0):
    # along-strike length (km) for given Mw, slip (m), seismogenic width (km)
    area_m2 = Mw_to_M0(Mw) / (mu * slip)
    return area_m2 / (width_km * 1e3) / 1e3

print("Required rupture length (slip = 15 m):")
for Mw in [8.0, 8.5, 9.0, 9.5]:
    for W in (100, 150):
        print(f"  Mw {Mw}, W={W:3d} km -> L = {rupture_length(Mw, width_km=W):5.0f} km")
print("\nCascadia: width ~127 km, length ~1000 km (Wirth et al. 2022, Table 1).")

An $M_w\,9$ needs a fault hundreds of kilometres long on a wide locked
zone. Widen the seismogenic zone and the required length drops; narrow it and even
a long rupture cannot reach $M\,9$. This is the mechanical reason a *geometric*
parameter — width — outranks slab age.

### Exercise 3 — Why steep margins stay small

The Mariana margin subducts very old lithosphere but at a steep dip, giving a
narrow seismogenic zone. Using `rupture_length`, find the along-strike length
needed for an $M_w\,9$ if the seismogenic width is only $W = 50$ km. Is that
plausible for the Mariana arc? Connect your answer to why the old Ruff–Kanamori
recipe (which would rank old-plate Mariana high) fails.

In [ ]:
# Exercise 3: your code here.
# Hint: rupture_length(9.0, width_km=50.0)


## 6. Worked classification: Chile, Mariana, Cascadia

Pull Maule (Chile) and Cascadia from the Wirth table and contrast them with
Mariana. Mariana is **not** in Wirth Table 1 (it has hosted no great earthquake);
the values below are representative literature figures, flagged as such, included
only for the contrast.

In [ ]:
chile = zones[zones.name == "Maule 2010"].iloc[0]
casc  = zones[zones.name == "Cascadia 1700"].iloc[0]
# Mariana: representative values (NOT from Wirth Table 1 - no great earthquake).
mariana = {"age_Ma": 155, "dip_deg": 78, "width_km": 55, "Mw": 7.3}

print(f"{'segment':16s}{'age':>6}{'dip':>6}{'width':>8}{'Mw':>6}   recipe vs reality")
print(f"{'Maule (Chile)':16s}{chile.age_Ma:>6.0f}{chile.dip_deg:>6.0f}"
      f"{chile.width_km:>8.0f}{chile.Mw:>6.1f}   high recipe, IS great (M8.8/9.5)")
print(f"{'Cascadia':16s}{casc.age_Ma:>6.0f}{casc.dip_deg:>6.0f}"
      f"{casc.width_km:>8.0f}{casc.Mw:>6.1f}   LOW recipe (young,slow), yet M~9")
print(f"{'Mariana*':16s}{mariana['age_Ma']:>6}{mariana['dip_deg']:>6}"
      f"{mariana['width_km']:>8}{mariana['Mw']:>6.1f}   high recipe (old), NO great EQ")
print("\n* Mariana: representative values, not in Wirth Table 1.")

The three rows make the lecture's point with the real data:

- **Chile (Maule)** — shallow dip ($22^\circ$), moderate width (105 km): fits the
  old recipe *and* is a great-earthquake margin.
- **Cascadia** — young and slow, so the old recipe rates it low, yet its shallow
  dip ($11^\circ$) and wide seismogenic zone (127 km) put it firmly in
  great-earthquake territory by the *geometric* controls — and it ruptured
  $M\sim 9$ in 1700.
- **Mariana** — old plate (the old recipe would rank it high) but a steep dip and
  narrow zone: no great earthquake.

Geometry (width, dip) explains all three; plate age explains none. That is the
thesis of the lecture, recovered from the data.

## Going Further

1. **The full Fig. 3.** Add the remaining Wirth parameters (coupling, roughness)
   as scatter panels and reproduce all eight correlation coefficients. Confirm
   that width and curvature top the ranking.
2. **Live geometry from Slab2.** Replace the Table 1 width and dip with values you
   compute directly from the Slab2 grids (`*_dep.grd`, `*_dip.grd`; USGS data
   release doi:10.5066/F7PV6JNV) along a trench-normal profile.
3. **Statistical power.** With only ~14–30 segments, correlation coefficients are
   unstable. Bootstrap-resample the rows and plot the distribution of
   `corr(Mw, width)`. How wide is it? This is why Wirth et al. stress that all the
   correlations have low statistical power.
4. **Cascadia recurrence.** The 1700 Japanese tsunami (Satake et al. 2003,
   doi:10.1029/2003JB002521) and Goldfinger's turbidite record constrain
   Cascadia's history. Sketch how a supercycle model would change a
   time-dependent hazard estimate relative to simple periodicity.

---

### References

- Wirth, E. A. et al. (2022). *Nat. Rev. Earth Environ.* 3, 125–140.
  doi:10.1038/s43017-021-00245-w  (US Govt work, public domain)
- Bletery, Q. et al. (2016). Mega-earthquakes rupture flat megathrusts.
  *Science* 354, 1027–1031. doi:10.1126/science.aag0482
- Hayes, G. P. et al. (2018). Slab2. *Science* 362, 58–61.
  doi:10.1126/science.aat4723; data doi:10.5066/F7PV6JNV
- Heuret, A. & Lallemand, S. (2005). *Phys. Earth Planet. Inter.* 149, 31–51.
  (SubMap database, http://submap.gm.univ-montp2.fr/)

*This notebook accompanies ESS 314 Lecture 28. Table 1 values are transcribed
from Wirth et al. (2022); the Mariana row in Section 6 is representative only.*